In [1]:
import json

##### Предполагается, что датасет в Label studio формате лежит в директории `../ls_deft_ru/`

In [2]:
# for label studio files
dev_file_ls = "../ls_deft_ru/dev.json"
train_file_ls = "../ls_deft_ru/train.json"

### Анализ данных

1. проверка, где нет predictions (то есть разметки)
2. проверка, где есть predictions и есть русский язык
3. подсчет случаев, когда определение и термин находятся в разных предложениях


**Проверять будем уже файлы в LS формате**

#### 1. проверка, где нет `predictions`

Если в одном элементе данных (одном предложении) в LS формате поле `predictions` равно пустому списку, значит никаких предварительных разметок не было.

In [3]:
# 1. проверка, где нет predictions

def check_where_no_predictions(input_file_ls):
    cnt = 0
    with open(input_file_ls, 'r') as f:
        label_studio_data = json.load(f)
    
    for obj in label_studio_data:
        if obj["predictions"] == []:
            cnt += 1
    print(f"In '{input_file_ls}' - empty predictions {cnt}/{len(label_studio_data)} .")

In [4]:
check_where_no_predictions(dev_file_ls)
check_where_no_predictions(train_file_ls)

In '../ls_deft_ru/dev.json' - empty predictions 847/1179 .


In '../ls_deft_ru/train.json' - empty predictions 17202/24184 .


#### 2. проверка, где есть `predictions` и есть русский язык (`text_rus`)

Если в одном элементе данных (одном предложении) в LS формате есть непустое `predictions` и в поле `data` есть непустой `text_rus`, значит этот элемент данных содержит разметку и русский язык.

In [5]:
# 2. проверка, где есть predictions и есть русский язык

def check_where_predictions_and_rustext(input_file_ls):
    cnt = 0
    with open(input_file_ls, 'r') as f:
        label_studio_data = json.load(f)
    
    for obj in label_studio_data:
        if obj["predictions"] != [] and obj["data"]["text_rus"] != "":
            cnt += 1
    print(f"In '{input_file_ls}' - where predictions and rus text {cnt}/{len(label_studio_data)} .")

In [6]:
check_where_predictions_and_rustext(dev_file_ls)
check_where_predictions_and_rustext(train_file_ls)

In '../ls_deft_ru/dev.json' - where predictions and rus text 332/1179 .
In '../ls_deft_ru/train.json' - where predictions and rus text 723/24184 .


#### 3. подсчет случаев, когда определение и термин находятся в разных предложениях

Если в одном элементе данных (одном предложении) в LS формате есть непустое `predictions` и в поле `data` есть непустой `text_rus`, значит этот элемент данных содержит разметку и русский язык.

Как я понимаю, в label studio задачи на разметку формируются по принципу одно предложение (одна строка в датасете) = одна задача на разметку.
Поэтому существует проблема. 
Если вот будет определение слова (Definition) в одном предложении, а сам термин (Term) в другом, то получается задача не будет отображаться корректно.

Поэтому я проверил случаи по следующим критериям:
1. я в рамках одного предложения собирал id лейблов (`label_ids`) и id связей (`relation_ids`) в множества.
2. Если для связи в предложении нет хоть одного id от лейбла (к которому `to_id` или от которого `from_id` связь идет), то эта **страница не загрузится**. 
    - я проверял это условием, что множество id в связях должно быть подмножеством у множества id лейблов
    - количество таких случаев собирается в переменную `cnt_not_rels`
3. И просто я так же посчитал случаи, где в целом не равны id лейблов и id связей
    - такие случаи я собрал в переменную `cnt_diff`, то есть где разные множества получаются
    - Наличие такого случая не всегда является проблемой для загрузки страницы, т.к. может быть что в предложении 2 термина и 1 определение, поэтому один термин используется для связи с определением, например, из следующего предложения.
    - то есть количество случаев из п.2 является подмножеством количества случаев п.3.



In [7]:
# 3. подсчет случаев, когда определение и термин находятся в разных предложениях

def check_term_and_rels_in_diff_strings(input_file_ls):
    cnt_diff = 0
    cnt_not_rels = 0
    cnt_predictions = 0     # подсчет тех предложений, где есть разметка

    with open(input_file_ls, 'r') as f:
        label_studio_data = json.load(f)
    
    for obj in label_studio_data:
        if obj["predictions"] != []:
            cnt_predictions += 1
            label_ids = set()
            relation_ids = set()
            for res in obj["predictions"][0]["result"]:

                if res['type'] == "labels":
                    label_ids.add(res['id'])
                elif res['type'] == "relation":
                    relation_ids.add(res['from_id'])
                    relation_ids.add(res['to_id'])

            if relation_ids != label_ids:
                cnt_diff += 1
            
            if not relation_ids.issubset(label_ids):
                cnt_not_rels += 1


            

    print(f"In '{input_file_ls}' - predictions where tag ids != relation ids - {cnt_diff}/{cnt_predictions} .")
    print(f"In '{input_file_ls}' - predictions where string has broken relations - {cnt_not_rels}/{cnt_predictions} .")


In [8]:
check_term_and_rels_in_diff_strings(dev_file_ls)
check_term_and_rels_in_diff_strings(train_file_ls)

In '../ls_deft_ru/dev.json' - predictions where tag ids != relation ids - 67/332 .
In '../ls_deft_ru/dev.json' - predictions where string has broken relations - 35/332 .
In '../ls_deft_ru/train.json' - predictions where tag ids != relation ids - 1515/6982 .
In '../ls_deft_ru/train.json' - predictions where string has broken relations - 902/6982 .


#### 4. проверка, где есть `predictions` и есть русский язык (`text_rus`) и предложение не сломается в Label studio


- Если в одном элементе данных (одном предложении) в LS формате есть непустое `predictions` и в поле `data` есть непустой `text_rus`, значит этот элемент данных содержит разметку и русский язык.
- потом проверяем его на случай из п.2 проверки 3 (предыдущая функция) 
и получаем еще из всех предложений с разметкой и русским языком еще те, которые среди них не сломаются в label studio

In [9]:
# 4. проверка, где есть predictions и есть русский язык
# + в этих связях определение и термин находятся в одном предложении
# то есть не сломается таск в label studio

def check_where_predictions_and_rustext_and_rels_not_broken(input_file_ls):
    cnt_predictions = 0
    cnt_predictions_not_broken_with_rus_text = 0

    with open(input_file_ls, 'r') as f:
        label_studio_data = json.load(f)
    
    for obj in label_studio_data:
        if obj["predictions"] != [] and obj["data"]["text_rus"] != "":
            cnt_predictions += 1

            label_ids = set()
            relation_ids = set()
            for res in obj["predictions"][0]["result"]:

                if res['type'] == "labels":
                    label_ids.add(res['id'])
                elif res['type'] == "relation":
                    relation_ids.add(res['from_id'])
                    relation_ids.add(res['to_id'])

            # если множество id связей является подмножеством для множества id лейблов, то таск не сломается
            if relation_ids.issubset(label_ids):    
                cnt_predictions_not_broken_with_rus_text += 1

    
    print(f"In '{input_file_ls}' - where predictions and rus text {cnt_predictions_not_broken_with_rus_text}/{cnt_predictions} .")

In [10]:
check_where_predictions_and_rustext_and_rels_not_broken(dev_file_ls)
check_where_predictions_and_rustext_and_rels_not_broken(train_file_ls)

In '../ls_deft_ru/dev.json' - where predictions and rus text 297/332 .
In '../ls_deft_ru/train.json' - where predictions and rus text 630/723 .
